# 02b — Phụ Lục Indexing Pipeline
**Pipeline xử lý .doc/.docx → Chunk → Index vào Qdrant/ChromaDB**

Sơ đồ:
```
Data/raw/phuluc/{tt12,tt13,tt35,tt48}/*.doc(x)
  → Convert .doc → .docx  (LibreOffice headless)
  → Extract text           (python-docx)
  → Parse metadata         (folder/filename patterns)
  → Chunk                  (theo loại tài liệu)
  → LangChain Documents
  → Index vào Qdrant / ChromaDB
```

## 0. Cài đặt thư viện

In [ ]:
# Cài các thư viện cần thiết (bỏ qua nếu đã có trong môi trường)
# !pip install python-docx docx2txt tqdm langchain langchain-core sentence-transformers
# !pip install qdrant-client langchain-qdrant
# !pip install chromadb   # nếu dùng ChromaDB

## 1. Cấu hình đường dẫn

In [ ]:
import sys, os, logging
from pathlib import Path

# ── Thêm thư mục gốc dự án vào PYTHONPATH ──────────────────────────────────
PROJECT_ROOT = Path("/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Đường dẫn dữ liệu ─────────────────────────────────────────────────────
INPUT_ROOT     = PROJECT_ROOT / "Data" / "raw" / "phuluc"
CONVERTED_DIR  = PROJECT_ROOT / "Data" / "converted_docx" / "phuluc"
OUTPUT_DIR     = PROJECT_ROOT / "Data" / "chunks" / "phuluc"
VECTOR_DB_DIR  = PROJECT_ROOT / "Data" / "vector_db" / "phuluc"

for d in [CONVERTED_DIR, OUTPUT_DIR, VECTOR_DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("INPUT_ROOT     :", INPUT_ROOT)
print("CONVERTED_DIR  :", CONVERTED_DIR)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("VECTOR_DB_DIR  :", VECTOR_DB_DIR)

In [ ]:
# Cấu hình logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("phuluc_indexing")

## 2. Kiểm tra LibreOffice

In [ ]:
import shutil, subprocess

soffice = shutil.which("soffice")
if soffice:
    result = subprocess.run([soffice, "--version"], capture_output=True, text=True)
    print(f"✅ LibreOffice tìm thấy: {soffice}")
    print("   Version:", result.stdout.strip())
else:
    print("⚠️  LibreOffice KHÔNG tìm thấy trong PATH.")
    print("   Cài đặt bằng: sudo apt-get install libreoffice")
    print("   Pipeline sẽ fallback sang docx2txt.")

## 3. Chạy Pipeline

In [ ]:
from source.doc_loader.pipeline import DocRAGPipeline

pipeline = DocRAGPipeline(
    input_root    = str(INPUT_ROOT),
    output_dir    = str(OUTPUT_DIR),
    converted_dir = str(CONVERTED_DIR),
)

# save_json=True → lưu chunks ra OUTPUT_DIR/<folder>_<name>.json để debug
all_documents = pipeline.run(save_json=True)

In [ ]:
# ── Xem mẫu document đầu tiên ─────────────────────────────────────────────
if all_documents:
    doc = all_documents[0]
    print("=== page_content (200 ký tự đầu) ===")
    print(doc.page_content[:200])
    print("\n=== metadata ===")
    for k, v in doc.metadata.items():
        print(f"  {k:20s}: {v}")

## 4. Thống kê chunk theo loại tài liệu

In [ ]:
from collections import Counter
import pandas as pd

stats = Counter(
    d.metadata.get("loai_tai_lieu", "unknown") for d in all_documents
)
by_folder = Counter(
    d.metadata.get("folder", "unknown") for d in all_documents
)

print("=== Chunks theo loại tài liệu ===")
for k, v in stats.most_common():
    print(f"  {k:25s}: {v:4d} chunks")

print("\n=== Chunks theo folder ===")
for k, v in by_folder.most_common():
    print(f"  {k:10s}: {v:4d} chunks")

## 5a. Index vào Qdrant (Recommended)

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_qdrant import QdrantVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "phuluc_documents"
EMBED_MODEL     = "keepitreal/vietnamese-sbert"
VECTOR_SIZE     = 768

# ── Embedding model ────────────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# ── Kết nối Qdrant local ───────────────────────────────────────────────────
# Option A: Qdrant chạy qua Docker (localhost:6333)
client = QdrantClient(host="localhost", port=6333)

# Option B: Qdrant lưu local (không cần Docker)
# client = QdrantClient(path=str(VECTOR_DB_DIR))

# ── Tạo collection nếu chưa có ────────────────────────────────────────────
existing = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME not in existing:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    print(f"✅ Đã tạo collection: {COLLECTION_NAME}")
else:
    print(f"ℹ️  Collection đã tồn tại: {COLLECTION_NAME}")

In [ ]:
# ── Batch index ────────────────────────────────────────────────────────────
BATCH_SIZE = 32

vectordb_qdrant = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

for i in range(0, len(all_documents), BATCH_SIZE):
    batch = all_documents[i : i + BATCH_SIZE]
    vectordb_qdrant.add_documents(batch)
    done = min(i + BATCH_SIZE, len(all_documents))
    print(f"Indexed {done}/{len(all_documents)} documents...")

print("\n✅ Index Qdrant hoàn thành!")

## 5b. Index vào ChromaDB (Thay thế)

In [ ]:
# from langchain_community.vectorstores import Chroma
# from langchain_community.embeddings import HuggingFaceEmbeddings

# embeddings = HuggingFaceEmbeddings(
#     model_name="keepitreal/vietnamese-sbert",
#     encode_kwargs={"normalize_embeddings": True},
# )

# vectordb_chroma = Chroma(
#     collection_name="phuluc_documents",
#     embedding_function=embeddings,
#     persist_directory=str(VECTOR_DB_DIR / "chroma"),
# )

# BATCH_SIZE = 50
# for i in range(0, len(all_documents), BATCH_SIZE):
#     batch = all_documents[i:i+BATCH_SIZE]
#     vectordb_chroma.add_documents(batch)
#     print(f"Indexed {min(i+BATCH_SIZE, len(all_documents))}/{len(all_documents)}")

# vectordb_chroma.persist()
# print("[DONE] Index ChromaDB hoàn thành!")

## 6. Test Retrieval

In [ ]:
def test_phuluc_retrieval(vectordb, k: int = 3):
    """Test retrieval với các câu query mẫu."""
    test_queries = [
        "biên bản vụ việc tai nạn giao thông",
        "mẫu số báo cáo kết quả khám nghiệm",
        "quy chuẩn kỹ thuật quốc gia 2024",
        "kế hoạch xác minh tai nạn",
        "quyết định phân công cán bộ",
        "thông báo kết quả xét nghiệm",
        "sổ theo dõi tai nạn giao thông",
    ]

    for query in test_queries:
        print(f"\n🔍 Query: {query}")
        try:
            results = vectordb.similarity_search(query, k=k)
            for r in results:
                meta = r.metadata
                loai    = meta.get("loai_tai_lieu", "?")
                folder  = meta.get("folder", "?")
                ten     = meta.get("ten_van_ban", "?")[:50]
                chunk_i = meta.get("chunk_index", "?")
                strategy = meta.get("chunk_strategy", "?")
                print(f"  → [{loai}] {ten}")
                print(f"     folder={folder} | chunk={chunk_i} | strategy={strategy}")
        except Exception as e:
            print(f"  ⚠️  Lỗi: {e}")

# Dùng với Qdrant:
test_phuluc_retrieval(vectordb_qdrant)

# Dùng với ChromaDB:
# test_phuluc_retrieval(vectordb_chroma)

## 7. Unit test nhanh cho từng module

In [ ]:
# ── Test metadata_parser ───────────────────────────────────────────────────
from source.doc_loader.metadata_parser import parse_metadata_from_path
import json

test_paths = [
    "/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/Data/raw/phuluc/tt13/1.2. Biên bản vụ việc.doc",
    "/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/Data/raw/phuluc/tt13/1.6. Kế hoạch xác minh vụ TNGT A4.doc",
    "/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/Data/raw/phuluc/tt48/01. QCVN 04-2024.doc",
    "/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/Data/raw/phuluc/tt35/Phu luc.doc",
    "/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/Data/raw/phuluc/tt13/1.18. Mẫu số 01 phiếu chuyển kết quả từ TBKTNV.docx",
]

for tp in test_paths:
    meta = parse_metadata_from_path(tp)
    print(f"\nFile: {meta['filename']}")
    print(f"  loai_tai_lieu : {meta['loai_tai_lieu']}")
    print(f"  so_thong_tu   : {meta['so_thong_tu']}")
    print(f"  so_hieu       : {meta['so_hieu']}")
    print(f"  so_phu_luc    : {meta['so_phu_luc']}")

In [ ]:
# ── Test splitter với text mẫu ─────────────────────────────────────────────
from source.doc_loader.splitter import AdminDocumentSplitter

splitter = AdminDocumentSplitter()

sample_text = """
I. Mục đích - Yêu cầu
Kế hoạch này nhằm xác minh vụ tai nạn giao thông xảy ra ngày 01/01/2024.
Các cán bộ được phân công có trách nhiệm hoàn thành trong vòng 07 ngày.

II. Nội dung công tác
1. Thu thập chứng cứ tại hiện trường vụ tai nạn.
2. Lấy lời khai các nhân chứng liên quan.
3. Trưng cầu giám định kỹ thuật xe ô tô.

III. Tổ chức thực hiện
Cán bộ Nguyễn Văn A chịu trách nhiệm chính.
"""

meta_ke_hoach = {"loai_tai_lieu": "ke_hoach"}
chunks = splitter.split(sample_text, meta_ke_hoach)
print(f"ke_hoach → {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: [{c['strategy']}] {c['text'][:80]}...")